# Customer Support QA — Judge Benchmark

semantix (local NLI) vs. Groq Llama 3.3 70B vs. Gemini 2.5 Flash (proxy-ground-truth) vs. Gemini 2.5 Pro (verification slice).

Raw rows: `results/raw.csv`. Summary: `results/summary.md`.

In [ ]:
import pandas as pd
from benchmarks.common.metrics import pearson_r, cohen_kappa_binary
df = pd.read_csv('results/raw.csv')
df.head()

## Latency & cost

In [ ]:
import matplotlib.pyplot as plt
ok = df[df.error.isna()]
ok.groupby('judge')['latency_ms'].mean().sort_values().plot.barh(title='Mean latency (ms)')
plt.tight_layout(); plt.show()

## Agreement with Gemini 2.5 Flash (proxy-ground-truth)

In [ ]:
agreement = df[df.experiment == 'agreement'].copy()
piv = agreement.pivot_table(index='example_id', columns='judge', values='score')
ref = 'gemini-2.5-flash'
for judge in [c for c in piv.columns if c != ref]:
    r = pearson_r(piv[judge].tolist(), piv[ref].tolist())
    print(f'{judge:40s} Pearson r vs {ref} = {r:.3f}')

## Verification slice — Flash vs. Pro correlation

In [ ]:
slice_ids = sorted(agreement[agreement.judge == 'gemini-2.5-pro']['example_id'].unique())
slice_piv = agreement[agreement.example_id.isin(slice_ids)].pivot_table(
    index='example_id', columns='judge', values='score')
r = pearson_r(slice_piv['gemini-2.5-flash'].tolist(), slice_piv['gemini-2.5-pro'].tolist())
print(f'Flash<->Pro Pearson r on {len(slice_ids)}-example slice: {r:.3f}')

## Optimization-impact: BestOfN win-rate

In [ ]:
opt = df[df.experiment == 'optimization'].copy()
opt['reward_judge'] = opt['judge'].str.split('__reward::').str[1]
piv = opt.pivot_table(index='example_id', columns='reward_judge', values='score')
wins = (piv['semantix'] > piv['groq-llama-3.3-70b']).sum()
losses = (piv['semantix'] < piv['groq-llama-3.3-70b']).sum()
ties = len(piv) - wins - losses
print(f'semantix wins: {wins}, groq wins: {losses}, ties: {ties}')